In [ ]:
# this is an example script for dataprep for one rep (rep6) 
# it needs to be adjusted and rerun for every rep that contains multiple timepoints (reps 5, 6, 7, 8, 9, 10). 
# at the end, the 5 concatenated files for all the reps were combined into one (by copy-paste in excel) 
# the combined file is made available and imported in the script for plotting fig 2

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import sys
import os
import seaborn as sns
from scipy import stats as scipystats
import statannot
from statannot import add_stat_annotation
from scipy.stats import mannwhitneyu, normaltest
import math


pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 3)
pd.set_option('display.expand_frame_repr', False)

# ***assign path to folder containing coulter-counter output files for multiple time points*** 

In [ ]:
exp_path = 'C:/Users/yagya.chadha/Desktop/Yagya/Coulter Counter/07.07.2022/SCD to SCGE/rep 6/excel files'

filenames = os.listdir(exp_path)
#print (filenames)


# ***open each file and reformat the weird coulter-counter output table to make it a usable df*** 
# ***calculate bin cell vol from bin diameter*** 

In [ ]:
# at the beginning of these experiments, we measured data from two size ranges on the coulter-counter but later on decided to only use one size-range.
# hence, in all of the excel files that we use here, the size range is always 10 to 250 fl
df_li = []
keys = []
for file in filenames:
    file_path = os.path.join(exp_path, file)
    print(file)
    df_file = pd.read_csv(file_path, sep='\t',encoding= 'unicode_escape', skiprows =[i for i in range(0,104)])
    print(df_file)

    new_df= df_file.iloc[:,0:5]
    print(new_df.columns[2])
    strain_name = new_df.columns[2]
    if "MMY" in strain_name:
        new_df['strain'] = "MMY"
    else:
        new_df['strain'] = "whi5_bck2_dd"

    if (new_df.iloc[4,1] == "1.34816"):
        new_df['sizerange'] = "10to250"
    elif (new_df.iloc[4,1]) == "1.07015":
        new_df['sizerange'] = "5to164"
    new_df['filename'] = strain_name
    new_df.columns = ['Bin Number', 'Bin Diameter (Lower) - um', 'bin_freq', 'Diff. Number/ mL', 'Diff. Volume %', 'strain', 'sizerange', 'filename']
    new_df.drop([0, 1,2], inplace = True)
    new_df.reset_index(inplace = True, drop = True)


    list = [0,1,5,6,7]
    new_df_2= df_file.iloc[:,list]
    strain_name = new_df_2.columns[2]
    if "MMY" in strain_name:
        new_df_2['strain'] = "MMY"
    else:
        new_df_2['strain'] = "whi5_bck2_dd"

    if (new_df_2.iloc[4,1] == "1.34816"):
        new_df_2['sizerange'] = "10to250"
    elif (new_df_2.iloc[4,1]) == "1.07015":
        new_df_2['sizerange'] = "5to164"
    new_df_2['filename'] = strain_name
    new_df_2.columns = ['Bin Number', 'Bin Diameter (Lower) - um', 'bin_freq', 'Diff. Number/ mL', 'Diff. Volume %', 'strain', 'sizerange', 'filename']
    new_df_2.drop([0, 1,2], inplace = True)
    new_df_2.reset_index(inplace = True, drop = True)

    timepoint_df = pd.concat([new_df, new_df_2])

    #  create amazing df
    df_li.append(timepoint_df)


big_df = pd.concat(df_li).reset_index(drop = True)
big_df['Bin Diameter (Lower) - um'] = (big_df['Bin Diameter (Lower) - um']).astype(float)
big_df['bin_freq'] = (big_df['bin_freq']).astype(float)
big_df['Bin radius um'] = (big_df['Bin Diameter (Lower) - um'])/2
big_df['Bin cell_vol_fl'] = (4/3)*(math.pi)*((big_df['Bin radius um'])**3)

# ***calclate mean binsize (vol) for each file*** 

In [ ]:
list_2 = []
for filename in big_df['filename'].unique():
    file_df = big_df[big_df['filename'] == filename]

    file_df = file_df.assign(timepoint=(file_df['filename']).str[:11])

    file_df['rep'] = 6
    file_df['binsize_vol_fl'] = np.nan
    for row in file_df.index:
        if row != file_df.index[-1]:
            file_df['binsize_vol_fl'].loc[row] = (file_df['Bin cell_vol_fl'].loc[row + 1]) - (file_df['Bin cell_vol_fl'].loc[row])
    #print(file_df)
    file_df['mean_binsize_vol'] = np.nanmean(file_df['binsize_vol_fl'])
    #print(file_df)
    list_2.append(file_df)
final_df = pd.concat(list_2).reset_index(drop = True)

# ***assign time_since_switch_hours for every timepoint*** 

In [ ]:
final_df['time_since_switch_hrs'] = np.nan
for row in final_df.index:
    if ((final_df['timepoint'].loc[row] == "0reading_MM")|(final_df['timepoint'].loc[row] == "0reading_YC")):
        final_df['time_since_switch_hrs'].loc[row] = 0
    elif final_df['timepoint'].loc[row] == "day1_1600pm":
        final_df['time_since_switch_hrs'].loc[row] = 2.0
    elif final_df['timepoint'].loc[row] == "day1_1800pm":
        final_df['time_since_switch_hrs'].loc[row] = 4.0
    elif final_df['timepoint'].loc[row] == "day1_2000pm":
        final_df['time_since_switch_hrs'].loc[row] = 6.0
    elif final_df['timepoint'].loc[row] == "day1_2200pm":
        final_df['time_since_switch_hrs'].loc[row] = 8.0
    elif final_df['timepoint'].loc[row] == "day2_0830am":
        final_df['time_since_switch_hrs'].loc[row] = 18.5
    elif final_df['timepoint'].loc[row] == "day2_0930am":
        final_df['time_since_switch_hrs'].loc[row] = 19.5
    elif final_df['timepoint'].loc[row] == "day2_1030am":
        final_df['time_since_switch_hrs'].loc[row] = 20.5
    elif final_df['timepoint'].loc[row] == "day2_1130am":
        final_df['time_since_switch_hrs'].loc[row] = 21.5
    elif final_df['timepoint'].loc[row] == "day2_1230pm":
        final_df['time_since_switch_hrs'].loc[row] = 22.5
    elif final_df['timepoint'].loc[row] == "day2_1330pm":
        final_df['time_since_switch_hrs'].loc[row] = 23.5
    elif final_df['timepoint'].loc[row] == "day2_1430pm":
        final_df['time_since_switch_hrs'].loc[row] = 24.5
    elif final_df['timepoint'].loc[row] == "day2_1600pm":
        final_df['time_since_switch_hrs'].loc[row] = 26
    elif final_df['timepoint'].loc[row] == "day2_1830pm":
        final_df['time_since_switch_hrs'].loc[row] = 28.5


final_df['time_since_switch_hrs'] = (final_df['time_since_switch_hrs']).astype(float)

# ***calculate true mean cell volume, stdev and CV for each timepoint fom the histogram*** 

In [ ]:
list_3 = []
for strain in final_df['strain'].unique():
    strain_df = final_df[final_df['strain'] == strain]
    for timepoint in strain_df['timepoint'].unique():
        timepoint_df = strain_df[strain_df['timepoint'] == timepoint]
        sum_diff_no = np.sum(timepoint_df['bin_freq'])
        timepoint_df['no_percent'] = ((timepoint_df['bin_freq'])/sum_diff_no)*100
        timepoint_df['bin_midpoint_cell_vol'] = ((timepoint_df['Bin cell_vol_fl'])+((timepoint_df['mean_binsize_vol'])/2))
        timepoint_df['bin_midpoint_x_bin_freq'] = (timepoint_df['bin_midpoint_cell_vol'])*(timepoint_df['bin_freq'])
        timepoint_df['true_mean_cell_vol_fl'] = (np.sum(timepoint_df['bin_midpoint_x_bin_freq']))/sum_diff_no
        timepoint_df['midpt_minus_mean_wholesquared'] = ((timepoint_df['bin_midpoint_cell_vol']) - (timepoint_df['true_mean_cell_vol_fl']))**2
        timepoint_df['midpt_minus_mean_wholesquared_x_freq'] = ((timepoint_df['midpt_minus_mean_wholesquared'])*(timepoint_df['bin_freq']))
        timepoint_df['summation_midpt_minus_mean_wholesquared_x_freq'] = np.sum(timepoint_df['midpt_minus_mean_wholesquared_x_freq'])
        timepoint_df['stdev'] = np.sqrt((timepoint_df['summation_midpt_minus_mean_wholesquared_x_freq'])/(sum_diff_no-1))
        timepoint_df['CV'] = (timepoint_df['stdev'])/(timepoint_df['true_mean_cell_vol_fl'])

        list_3.append(timepoint_df)
final_df = pd.concat(list_3).reset_index(drop=True)

# ***save one file per rep and combine all reps manually in excel*** 

In [ ]:
final_df.to_csv(path_or_buf = 'C:/Users/yagya.chadha/Desktop/Yagya/Coulter Counter/07.07.2022/SCD to SCGE/final_analysed_df_070722_rep6.csv')